# Week 4: Transfer Learning, BERT (Seminar)

### Using pretrained transformers (for fun, profit and 1 point)

There are many toolkits that let you access pretrained transformer models (like we used pretrained embeddings earlier), but the most powerful and convenient by far is 🤗[`huggingface/transformers`](https://github.com/huggingface/transformers). In this week's practice, you'll learn how to download, apply and modify pretrained transformers for a range of tasks. Buckle up, we're going in!


__Pipelines:__ if all you want is to apply a pretrained model, you can do that in one line of code using pipeline. Huggingface/transformers has a selection of pre-configured pipelines for masked language modelling, sentiment classification, question aswering, etc. ([see full list here](https://huggingface.co/transformers/main_classes/pipelines.html))

A typical pipeline includes:
* pre-processing, e.g. tokenization, subword segmentation
* a backbone model, e.g. bert finetuned for classification
* output post-processing

Let's see it in action:

In [45]:
import transformers

In [46]:
sentiment_clf = transformers.pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

sentiment_clf(["transformers library can be really useful!", "YSDA midterm is soon"])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9959487915039062},
 {'label': 'NEGATIVE', 'score': 0.986364483833313}]

In [3]:
transformers.pipelines.SUPPORTED_TASKS.keys()

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'table-question-answering', 'document-question-answering', 'fill-mask', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'keypoint-matching', 'any-to-any'])

But how can we find out which model is suitable for chosen task in such a big models space?

Option 1: Using search and filters in [web](https://huggingface.co/models) (user-friendly)

Option 2: Using `huggingface_hub` library to access API from Python (if you want to automate some process)


In [4]:
import huggingface_hub

In [5]:
some_model = next(huggingface_hub.list_models())

some_model

ModelInfo(id='Cloudflare/clef', author=None, base_models=None, card_data=None, children_model_count=None, config=None, created_at=datetime.datetime(2026, 9, 30, 21, 15, 35, tzinfo=datetime.timezone.utc), disabled=None, downloads=5416, downloads_all_time=None, eval_results=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, last_modified=None, library_name='transformers', likes=1532, mask_token=None, model_index=None, pipeline_tag='image-text-to-text', private=False, resource_group=None, safetensors=None, security_repo_status=None, sha=None, siblings=None, spaces=None, tags=['transformers', 'safetensors', 'qwen3_5', 'image-text-to-text', 'clef', 'cloudflare', 'systemone', 'qwen3.8', 'post-train', 'image-text-to-typed-output', 'multimodal', 'structured-output', 'classification', 'custom-code', 'conversational', 'base_model:Qwen/Qwen3.8-27B', 'base_model:finetune:Qwen/Qwen3.8-27B', 'license:apache-2.0', 'endpoints_compatible', 'region:us'], transformers_info=None

In [6]:
filter = (
    "sentiment-analysis",
    "pytorch",
    "ru",
)

filtered_models = huggingface_hub.list_models(
    filter=filter,
    sort="downloads",
    limit=10,
)

print(f"Filtered by {filter}:")
for model in filtered_models:
    print(f"- https://huggingface.co/{model.id} ({model.downloads} downloads, {model.likes} likes)")

Filtered by ('sentiment-analysis', 'pytorch', 'ru'):
- https://huggingface.co/seara/rubert-base-cased-russian-sentiment (129722 downloads, 12 likes)
- https://huggingface.co/yangheng/deberta-v3-base-absa-v1.1 (52812 downloads, 74 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-sentiment (18947 downloads, 35 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-emotion-detection-ru-go-emotions (9838 downloads, 4 likes)
- https://huggingface.co/r1char9/rubert-base-cased-russian-sentiment (7860 downloads, 13 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-emotion-detection-cedr (998 downloads, 1 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-emotion-detection-ru-go-emotions (729 downloads, 13 likes)
- https://huggingface.co/yangheng/deberta-v3-large-absa-v1.1 (457 downloads, 26 likes)
- https://huggingface.co/apkonsta/finrubert (177 downloads, 3 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-emotion-detection-cedr (62 downloads,

Imagine the situation when you have a long text to read and a lack of time. Luckily, you've got an option to use one of pipelines! But which one?...

**Task 1 (0.5 points)**
- Find a suitable pipeline and model for text below
- Apply model to long text to get a short one
- Pretty-print the result and give an opinion if short text is good or not



In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [14]:
long_text = """
The widespread adoption of remote work, accelerated by global events in the early 2020s, has triggered a significant and likely permanent shift in how we think about the workplace. This transition away from the traditional central office is having profound and multifaceted effects on urban economies, reshaping everything from commercial real estate to local small businesses.

One of the most immediate and visible impacts has been on the commercial real estate sector. With companies downsizing their physical footprints or adopting fully remote models, demand for office space has plummeted. This has led to rising vacancy rates, downward pressure on commercial rent prices, and a re-evaluation of the financial viability of large office buildings. City governments, which often rely heavily on property taxes from these high-value commercial properties, are now facing substantial budget shortfalls.

Furthermore, the daily rhythm of city centers has changed dramatically. The decline in the number of commuters has had a ripple effect on local businesses that once thrived on their patronage. Lunchtime cafes, after-work bars, dry cleaners, and public transit systems have all experienced a significant drop in revenue. This "doughnut effect" describes a phenomenon where the economic activity hollows out in the city center and increases in suburban residential areas as people work from home and spend their money locally.

However, it's not all negative. This shift also presents new opportunities. Some urban planners see a chance to repurpose vacant office buildings into much-needed residential housing, which could help address housing shortages and revitalize neighborhoods by creating 24/7 communities. Additionally, the ability to work remotely has spurred a reversal of rural depopulation in some regions, as professionals seek a better quality of life outside of major metropolitan areas, potentially distributing economic growth more evenly.

In conclusion, the remote work revolution is fundamentally restructuring urban economies. While it presents serious challenges to established systems like commercial real estate and downtown commerce, it also opens the door to innovative urban renewal and a more geographically dispersed economic landscape. The long-term effects will depend on how effectively cities and businesses can adapt to this new, more flexible paradigm.
"""
sum_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

inputs = sum_tokenizer(long_text, return_tensors="pt", truncation=True, max_length=1024)
summary_ids = sum_model.generate(**inputs, max_length=80, min_length=20, do_sample=False)

short_text = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [17]:
print(short_text)

The widespread adoption of remote work has triggered a significant and likely permanent shift in how we think about the workplace. This transition away from the traditional central office is having profound and multifaceted effects on urban economies. The decline in the number of commuters has had a ripple effect on local businesses.


In [15]:
assert len(long_text) / len(short_text) > 5, "Too long, didn't read"

One of possible semi-supervised tasks used while BERT training is Masked Language Modeling. So our model have some text prediction capabilities!



In [16]:
mlm_model = transformers.pipeline(
    task="fill-mask",
    model="bert-base-cased"
)

mlm_model("My name is [MASK] Shady!")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[{'score': 0.0911569893360138,
  'token': 22191,
  'token_str': 'Slim',
  'sequence': 'My name is Slim Shady!'},
 {'score': 0.025677727535367012,
  'token': 2791,
  'token_str': 'Captain',
  'sequence': 'My name is Captain Shady!'},
 {'score': 0.023223936557769775,
  'token': 3056,
  'token_str': 'Miss',
  'sequence': 'My name is Miss Shady!'},
 {'score': 0.016291704028844833,
  'token': 4479,
  'token_str': 'Jimmy',
  'sequence': 'My name is Jimmy Shady!'},
 {'score': 0.013304632157087326,
  'token': 13960,
  'token_str': 'Mister',
  'sequence': 'My name is Mister Shady!'}]

In order to make result more readable we can just take top-1 result:

In [18]:
mlm_model("My name is [MASK] Shady!")[0]["sequence"]

'My name is Slim Shady!'

**Task 2 (0.5 points)**
- Using BERT's ability to solve MLM task, find out answers on the following questions
- Perform some fact-checking, don't trust LLMs!

**Questions:**
- When YSDA was founded?
- Who invented radio first?
- What is the fifth Fibonacci number?

---

In [26]:
questions = [
    "YSDA was founded in [MASK].",
    "Radio was invented by [MASK].",
    "The fifth Fibonacci number is [MASK].",
]

for q in questions:
    print(q)
    for r in mlm_model(q)[:3]:
        print(f"   {r['sequence']}  (вероятность {r['score']:.2f})")

YSDA was founded in [MASK].
   YSDA was founded in India.  (вероятность 0.02)
   YSDA was founded in Japan.  (вероятность 0.02)
   YSDA was founded in c.  (вероятность 0.02)
Radio was invented by [MASK].
   Radio was invented by Fr.  (вероятность 0.05)
   Radio was invented by him.  (вероятность 0.04)
   Radio was invented by Germans.  (вероятность 0.02)
The fifth Fibonacci number is [MASK].
   The fifth Fibonacci number is zero.  (вероятность 0.03)
   The fifth Fibonacci number is unknown.  (вероятность 0.02)
   The fifth Fibonacci number is lost.  (вероятность 0.02)


### The building blocks of a pipeline

Huggingface also allows you to access its pipelines on a lower level. There are two main abstractions for you:
* `Tokenizer` - converts from strings to token ids and back
* `Model` - a PyTorch `nn.Module` with pretrained weights

You can use such models as part of your regular PyTorch code: insert it as a layer in your model, apply to a batch of data, backpropagate, optimize, etc.

In [19]:
tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
model = transformers.AutoModel.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
lines = [
    "Luke, I am your father.",
    "Life is what happens when you're busy making other plans.",
    "I have no idea what pneumonoultramicroscopicsilicovolcanoconiosis is."
]

tokens_info = tokenizer(lines, padding=True, truncation=True, return_tensors="pt")
print("Tokenized:")
print(tokens_info)

print("\nDetokenized:")
for i in range(3):
    print(tokenizer.decode(tokens_info['input_ids'][i]))

Tokenized:
{'input_ids': tensor([[  101,  5355,  1010,  1045,  2572,  2115,  2269,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2166,  2003,  2054,  6433,  2043,  2017,  1005,  2128,  5697,
          2437,  2060,  3488,  1012,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1045,  2031,  2053,  2801,  2054,  1052,  2638,  2819, 17175,
         11314,  6444,  2594,  7352, 26461, 27572, 11261,  6767, 15472,  6761,
          8663, 10735,  2483,  2003,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]]), 'attention_mask': tensor([[1, 1,

You can see some special tokens appeared besides our original text. They are usually used to give model some additional information, so model treats them in individual way.

You can list all special tokens used by tokenizer (moreover, you can add your own special tokens, but make sure you will show them to your model while training):

In [21]:
tokenizer._special_tokens_map

{'bos_token': None,
 'eos_token': None,
 'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [22]:
tokenizer("First sentence", "Second sentence", return_token_type_ids=True)

{'input_ids': [101, 2034, 6251, 102, 2117, 6251, 102], 'token_type_ids': [0, 0, 0, 0, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

It's ineffective to put all possible tokens in vocabulary, but one also want to handle all possible text sequences instead of putting UNK everywhere.

WordPiece tokenization is here to help!

In [23]:
reversed_vocab = {token_id: token for token, token_id in tokenizer.vocab.items()}

In [24]:
for token_id in tokens_info["input_ids"][2]:
    print(reversed_vocab[token_id.item()], end=' ')

[CLS] i have no idea what p ##ne ##um ##ono ##ult ##ram ##ic ##ros ##copic ##sil ##ico ##vo ##lc ##ano ##con ##ios ##is is . [SEP] 

Now you can apply tokenized data with model.

Depending on your task, you can use different part of output. For example, `[CLS]`-token output can be obtained by `pooler_output` key in model output.

In [25]:
import torch

In [27]:
with torch.no_grad():
    out = model(**tokens_info)

print(out['pooler_output'])

tensor([[-0.8854, -0.4722, -0.9392,  ..., -0.8081, -0.6955,  0.8748],
        [-0.9297, -0.5161, -0.9334,  ..., -0.9017, -0.7492,  0.9201],
        [-0.6808, -0.1979, -0.7096,  ..., -0.6691, -0.4557,  0.7595]])


Transformers knowledge hub: https://huggingface.co/transformers/



---



### Visualizing BERT

Interpretability of models is one of key factors of understanding their behaviour.

Neural Networks are harder to interpret than Classic ML models, but still it's not impossible!

Remember Attention mechanism? It's human-understandable concept: look closely to tokens which are more valuable for context of the current one.

In [ ]:
!pip install bertviz

In [ ]:
from transformers import AutoTokenizer, AutoModel, utils
from bertviz import model_view, head_view

input_text = "Every time I try to interpret BERT model behaviour, I find new interesting patterns"
model = AutoModel.from_pretrained("bert-base-cased", output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

inputs = tokenizer.encode(input_text, return_tensors="pt")
outputs = model(inputs)
attention = outputs[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

model_view(attention, tokens)

In [ ]:
head_view(attention, tokens)

Another possible task for BERT training is Next Sentence Prediction.

How BERT's heads looks at tokens in that case?

In [ ]:
inputs = tokenizer.encode("I'm waiting for important call", "I can't go out right now", return_tensors="pt")
outputs = model(inputs)
attention = outputs[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

model_view(attention, tokens)

In [ ]:
head_view(attention, tokens)

It looks interesting, doesn't it?

If you want to find out more about attention patterns, you can refer to special "field" of science - [BERTology](https://huggingface.co/docs/transformers/main/en/bertology).



---



### Tuning pretrained transfomers (for your own task and 2 points)

Important benefit of using big models is their ability to adapt to various tasks without spending a lot of time and resources for full training.

You could've heard about backbone models in another ML tasks, when they're tuned using specific data.

It's possible to tune model's weights directly, but you also can freeze model, use its outputs as knowledge and then extract neccessary information using much smaller neural networks.

#### Introduction

Here's an example of tuned BERT base model for Named Entity Recognition (NER) task:

In [28]:
tokenizer = transformers.AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = transformers.AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

As you can see, there's an additional classifier besides original BERT content. That layer is used to predict NER-classes for each BERT's token output.

BERT is suitable for tuning for different tasks since it outputs token embeddings and the whole data embedding in `[CLS]`-token as well.

#### Data preparation

In [30]:
import datasets

In [31]:
dataset = datasets.load_dataset("lhoestq/conll2003")

dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  259kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [32]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [33]:
dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

Since BERT tokenization is different from the dataset's one, we need to fix that divergence.

**Task 3 (0.5 points)**
- Align dataset token labels to WordPiece tokens
- Handle special tokens as well

In [34]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [35]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [36]:
tokenized_dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'input_ids': [101,
  7270,
  22961,
  1528,
  1840,
  1106,
  21423,
  1418,
  2495,
  12913,
  119,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, -100, 0, -100]}

In [37]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Now dataset is ready to be used by BERT.

#### Model preparation

For our task we can use `AutoModelForTokenClassification`, which already provides required architecture with token classifier (e.g. classifier itself, class outputs).

You can handle these things by yourself: create PyTorch model class, init BERT model and Linear layer for classification, then override forward method and so on...

`AutoModelForTokenClassification` is chosen for the sake of simplicity, but it's still required for MLE to be capable of doing it with bare hands.

In [50]:
!git clone https://github.com/chakki-works/seqeval

Cloning into 'seqeval'...
remote: Enumerating objects: 808, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 808 (delta 6), reused 5 (delta 5), pack-reused 798 (from 2)
Receiving objects: 100% (808/808), 185.39 KiB | 4.41 MiB/s, done.
Resolving deltas: 100% (461/461), done.


In [51]:
!mv ьщвгду/seqeval .

In [54]:
from transformers import AutoModelForTokenClassification

id2label = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG", 5: "B-LOC", 6: "I-LOC", 7: "B-MISC", 8: "I-MISC"}
label2id = {label: id for id, label in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=9,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

#### Evaluation

Evaluation is crucial while writing papers or reporting your work results. Sometimes it can be tricky and own implementation can be buggy, so it usually preferred to calculate metrics using frameworks.

Let's prepare `compute_metrics` function for the following training loop:

In [53]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score
from seqeval.scheme import IOB2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    y_true = []
    y_pred = []
    for i in range(len(predictions)):
        y_true_sample = []
        y_pred_sample = []
        for j in range(len(predictions[i])):
            if labels[i][j] == -100:
                continue

            y_true_sample.append(id2label[int(labels[i][j])])
            y_pred_sample.append(id2label[int(predictions[i][j])])

        y_true.append(y_true_sample)
        y_pred.append(y_pred_sample)

    return {
        "precision": precision_score(y_true, y_pred, mode="strict", scheme=IOB2),
        "recall": recall_score(y_true, y_pred, mode="strict", scheme=IOB2),
        "f1": f1_score(y_true, y_pred, mode="strict", scheme=IOB2),
    }

#### Training

**Task 4 (0.5 points)**
- Choose proper hyperparameters for tuning the model
- Setup HF Trainer
- Check correctness using training results


In [64]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert-ner",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    report_to="none",

    learning_rate=0.0001,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
)

In [65]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Let's check metrics before training:

In [66]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)

Training Loss,Validation Loss,Step,Precision,Recall,F1
No log,0.138271,0,0.831385,0.805772,0.818378


{'eval_loss': 0.13827091455459595, 'eval_precision': 0.8313847278041652, 'eval_recall': 0.8057719546742209, 'eval_f1': 0.8183779895702211}


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Precision,Recall,F1
50,0.072525,0.124015,0.865843,0.855878,0.860832
100,0.091736,0.112307,0.865784,0.879426,0.872552
150,0.078238,0.098092,0.886222,0.882613,0.884414
200,0.058061,0.101947,0.871750,0.890581,0.881065
250,0.061174,0.092451,0.908550,0.891820,0.900107
300,0.050058,0.094900,0.891968,0.900496,0.896211
350,0.051331,0.103577,0.887955,0.898017,0.892958
400,0.046298,0.097883,0.903272,0.889518,0.896343
450,0.040229,0.090585,0.902935,0.904214,0.903574
500,0.022589,0.098021,0.904253,0.899610,0.901926


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)

Compare test metrics before and after training. Did we succeed?

**Task 5 (1 point)**
- Compare our model's result with `dslim/bert-base-NER`
- Try to improve our model's quality. Choose any option:
  - Play with training hyperparameters (batch_size, lr, epochs, etc.)
  - Apply some training techniques (warm-up, lr-scheduling, etc.)
  - Perform error analysis and find model's weak spots (this option doesn't require fixing them)
  - Your very own idea
- Write a small report (up to 5 steps, results and conclusions) on the work done in "Tuning pretrained transformers" part